# 06 COAD Model Report / 模型查看报告

这个 notebook 用来查看已经训练好的 COAD tumor vs normal RNA expression 模型。它不会修改数据库，也不会重新训练模型，只读取 `data/`、`models/`、`reports/` 里的结果。

**Research-only caveat / 重要限制：** tumor data comes from `bio_tcga`, normal data comes from `tcga_coad`; 两边可能有不同的数据处理流程，所以第一版结果只适合科研探索和报告展示，不能当作临床诊断模型。

## 1. Load Report Artifacts / 读取报告产物

In [ ]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "reports"

summary = json.loads((DATA_DIR / "data_summary.json").read_text(encoding="utf-8"))
metrics = pd.read_csv(REPORTS_DIR / "metrics_summary.csv")
important_genes = pd.read_csv(REPORTS_DIR / "important_genes.csv")
features = pd.read_parquet(DATA_DIR / "coad_tumor_normal_features.parquet")
labels = pd.read_csv(DATA_DIR / "coad_tumor_normal_labels.csv")
test_samples = pd.read_csv(DATA_DIR / "test_samples.csv")

display(Markdown(
    f"""
### Dataset Summary / 数据摘要

- Tumor samples: **{summary['tumor_samples']}**
- Normal samples: **{summary['normal_samples']}**
- Shared genes before low-variance filtering: **{summary['shared_genes']}**
- Model feature matrix shape: **{features.shape[0]} samples x {features.shape[1]} genes**
- Tumor source: `{summary['tumor_source']}`
- Normal source: `{summary['normal_source']}`
    """
))

## 2. Metrics / 模型效果指标

`accuracy` 看总体正确率；`balanced_accuracy` 更适合当前这种 normal 样本比较少的情况；`ROC-AUC` 和 `PR-AUC` 用来观察分类区分能力。

In [ ]:
metric_cols = [
    "model", "accuracy", "balanced_accuracy", "f1", "roc_auc", "pr_auc",
    "normal_precision", "normal_recall", "tumor_precision", "tumor_recall",
]
metrics[metric_cols].style.format({
    "accuracy": "{:.4f}",
    "balanced_accuracy": "{:.4f}",
    "f1": "{:.4f}",
    "roc_auc": "{:.4f}",
    "pr_auc": "{:.4f}",
    "normal_precision": "{:.4f}",
    "normal_recall": "{:.4f}",
    "tumor_precision": "{:.4f}",
    "tumor_recall": "{:.4f}",
})

## 3. Plots / 图表

下面三张图分别是 confusion matrix、ROC curve、PR curve。科研报告里建议配合 caveat 一起解释，不要只展示接近完美的指标。

In [ ]:
for filename in ["confusion_matrix.png", "roc_curve.png", "pr_curve.png"]:
    display(Markdown(f"### {filename}"))
    display(Image(filename=str(REPORTS_DIR / filename)))

## 4. What Is Inside The Saved Models? / 模型文件里有什么

每个 `.joblib` 文件保存了一个 sklearn `Pipeline`，里面包括 preprocessing step 和 model step。预测新样本时必须使用同一个 Pipeline，不能只拿最后的 classifier。

In [ ]:
model_rows = []
loaded_models = {}
for path in sorted(MODELS_DIR.glob("*.joblib")):
    payload = joblib.load(path)
    pipeline = payload["pipeline"]
    loaded_models[payload["model_name"]] = payload
    model_rows.append({
        "model": payload["model_name"],
        "file": path.name,
        "pipeline_steps": " -> ".join(pipeline.named_steps.keys()),
        "classifier": pipeline.named_steps["model"].__class__.__name__,
        "n_features": len(payload["features"]),
        "random_state": payload["random_state"],
    })

pd.DataFrame(model_rows)

## 5. Example Predictions / 看模型如何预测测试样本

`prediction` 是模型预测标签；`score_or_probability` 对 Logistic Regression 和 Random Forest 是 tumor probability，对 Linear SVM 是 decision score。

In [ ]:
classes = np.array(["normal", "tumor"])
test_features = features.loc[test_samples["sample_id"]]
preview = test_samples[["sample_id", "label", "source_schema", "sample_type"]].copy()

for name, payload in loaded_models.items():
    pipeline = payload["pipeline"]
    pred = pipeline.predict(test_features)
    preview[f"{name}_prediction"] = classes[pred]
    if hasattr(pipeline, "predict_proba"):
        preview[f"{name}_score_or_probability"] = pipeline.predict_proba(test_features)[:, 1]
    else:
        preview[f"{name}_score_or_probability"] = pipeline.decision_function(test_features)

preview.head(12)

## 6. Important Genes / 重要基因候选

这些 gene 是模型用来区分 tumor/normal 的 predictive features，不等于已经证明的癌症 driver genes。后续写科研报告时，需要做 literature review。

In [ ]:
important_genes.groupby("model").head(15)[[
    "model", "rank", "gene_symbol", "importance_score", "direction", "short_explanation"
]]

## 7. How To Explain This Model / 应该怎么看这个模型

- 第一眼看 `balanced_accuracy`，因为 normal 样本只有 41 个，比 tumor 少很多。
- 如果 Random Forest 结果是 1.0，不要直接说模型完美；更合理的解释是：当前数据来源差异可能让任务变得过于容易，需要外部验证。
- Logistic Regression 和 Linear SVM 的 coefficients 更适合做报告解释：positive coefficient means higher tumor score，negative coefficient means higher normal score。
- `important_genes.csv` 可以作为下一步查文献的候选列表，不要直接写成因果结论。
- 真正发表或严肃展示前，下一步应该做 TCGA-GTEx Toil / UCSC Xena 数据验证，检查模型是不是学到了 cancer biology，而不是 source/pipeline batch effect。